### 1º Trabalho Prático - ANADI
#### Grupo 8: 
##### Mariana Martins 1230679
##### Luna Gomes
##### Samara Miranda

### 1. Importação de Bibliotecas e Dados

In [4]:
!pip install pandas numpy matplotlib seaborn scipy statsmodels plotly streamlit openpyxl

### 2. Análise Exploratória (Estatística Descritiva)

In [4]:
import pandas as pd
import numpy as np

# 1. Carregar os dados (ajustado para os nomes dos ficheiros que enviaste)
# O parâmetro na_values converte automaticamente "N/D" em valores nulos (NaN)
ip_data = pd.read_excel('IP_data.xlsx', na_values='N/D')
ptd_data = pd.read_excel('PTD_data.xlsx', na_values='N/D')

# 2. Inspeção Rápida: Ver as primeiras linhas e os tipos de dados
print("--- INFO IP DATA ---")
print(ip_data.info())
print("\n--- INFO PTD DATA ---")
print(ptd_data.info())

# Visualizar as primeiras 5 linhas de cada um para confirmar a leitura
display(ip_data.head())
display(ptd_data.head())

--- INFO IP DATA ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9300 entries, 0 to 9299
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ano                           9300 non-null   int64  
 1   Mês                           9300 non-null   int64  
 2   Distrito                      9300 non-null   object 
 3   Concelho                      9300 non-null   object 
 4   Freguesia                     9300 non-null   object 
 5   Tipo de Lâmpada               9300 non-null   object 
 6   Luminárias                    9299 non-null   float64
 7   Lâmpadas por Luminária        9258 non-null   float64
 8   Lâmpadas                      9258 non-null   float64
 9   Potência Instalada Total (W)  8483 non-null   float64
 10  CodDistrito                   9300 non-null   int64  
 11  CodDistritoConcelho           9300 non-null   int64  
 12  coddistritoconcelhofreguesia  9300 non-nu

,Ano,Mês,Distrito,Concelho,Freguesia,Tipo de Lâmpada,Luminárias,Lâmpadas por Luminária,Lâmpadas,Potência Instalada Total (W),CodDistrito,CodDistritoConcelho,coddistritoconcelhofreguesia
0,2026,2,Leiria,Alvaiázere,Almoster,LED,716.0,715.0,715.0,29010.800171,10,1002,100201
1,2026,2,Leiria,Alvaiázere,Almoster,Outros/Não definido,23.0,23.0,23.0,225.000000,10,1002,100201
2,2026,2,Leiria,Alvaiázere,Almoster,Sódio,27.0,26.0,26.0,1890.000000,10,1002,100201
3,2026,2,Leiria,Alvaiázere,Pelmá,Mercúrio,1.0,1.0,1.0,50.000000,10,1002,100205
4,2026,2,Leiria,Alvaiázere,Alvaiázere,Mercúrio,3.0,3.0,3.0,150.000000,10,1002,100208


,Código de Instalação,Coordenadas Geográficas,Potência instalada [kVA],Nível de Utilização [%],Tipo Construtivo,Potência Contratada [kVA],Número de Clientes,Potência Geração [kW],Número de Clientes Produtores,CodDistritoConcelho,Concelho
0,1307D2012500,"41.1849010463209, -8.14785254356171",400,60%-79%,Cabine pré-fabricada,1052.69,131,NaN,<20,1307,Marco de Canaveses
1,1307D2012900,"41.0869934226954, -8.24673374651495",250,60%-79%,Cabine baixa integrada em edifício,775.20,115,NaN,<20,1307,Marco de Canaveses
2,1307D2013100,"41.1129221154249, -8.21276525945275",250,60%-79%,Cabine alta,1104.19,176,NaN,<20,1307,Marco de Canaveses
3,1307D2013800,"41.1329530486552, -8.13563404382479",50,+100%,Aéreo - AS,238.07,36,NaN,<20,1307,Marco de Canaveses
4,1307D2015100,"41.1228444565974, -8.11290068900831",250,0%-19%,Aéreo - AI,608.51,94,NaN,<20,1307,Marco de Canaveses


#### LIMPEZA DO PTD_DATA

In [5]:
# Função para converter os intervalos de utilização (ex: '60%-79%') em decimal
def limpar_utilizacao(valor):
    if pd.isna(valor) or valor == 'N/D':
        return np.nan
    valor = str(valor).replace('%', '').strip()
    if '-' in valor:
        # Pegamos no valor superior do intervalo (ex: '79') e dividimos por 100
        return float(valor.split('-')[-1]) / 100
    elif '+' in valor:
        # Para '+100%', assumimos 1.0 (100%)
        return 1.0
    try:
        return float(valor) / 100
    except:
        return np.nan

# Aplicar a limpeza
ptd_data['Utilizacao_Decimal'] = ptd_data['Nível de Utilização [%]'].apply(limpar_utilizacao)

#### LIMPEZA DO IP_DATA

In [6]:
# Criar a variável binária Is_Ineficiente (Sódio ou Mercúrio)
# Usamos .str.contains para evitar problemas com espaços ou maiúsculas
tipos_ineficientes = ['Sódio', 'Mercúrio']
ip_data['Is_Ineficiente'] = ip_data['Tipo de Lâmpada'].isin(tipos_ineficientes).astype(int)

# Converter Watts para kW e tratar valores em falta (NaN) com 0
ip_data['Potencia_kW'] = ip_data['Potência Instalada Total (W)'].fillna(0) / 1000

#### AGRUPAR POR CONCELHO E OVERVIEW FINAL

In [10]:
# Agrupar IP por Concelho (Somar potência total e potência ineficiente)
ip_concelho = ip_data.groupby(['CodDistritoConcelho', 'Concelho']).agg(
    P_IP_Total_kW=('Potencia_kW', 'sum'),
    P_IP_Inef_kW=('Potencia_kW', lambda x: x[ip_data.loc[x.index, 'Is_Ineficiente'] == 1].sum())
).reset_index()

# Agrupar PTD por Concelho (Capacidade total, Utilização média e Contagem de postos)
ptd_concelho = ptd_data.groupby(['CodDistritoConcelho']).agg(
    Cap_PTD_kVA=('Potência instalada [kVA]', 'sum'),
    Util_Media=('Utilizacao_Decimal', 'mean'),
    N_PTDs=('Código de Instalação', 'count')
).reset_index()

# --- 4. JUNÇÃO FINAL (MERGE) ---

df_final = pd.merge(ip_concelho, ptd_concelho, on='CodDistritoConcelho')

print("Novo dataset criado")
display(df_final.head())

Novo dataset criado


,CodDistritoConcelho,Concelho,P_IP_Total_kW,P_IP_Inef_kW,Cap_PTD_kVA,Util_Media,N_PTDs
0,101,Águeda,910.887701,244.320,105715,0.477593,388
1,102,Albergaria-a-Velha,451.711801,28.020,54540,0.469948,194
2,103,Anadia,657.071801,73.545,55628,0.543009,223
3,104,Arouca,585.974400,115.720,41884,0.527387,236
4,105,Aveiro,1055.192001,144.420,197485,0.475475,509


O que este código faz:
Normaliza a Utilização: Transforma o texto "60%-79%" no número 0.79. Isto permite-nos calcular a média de ocupação de cada concelho.

Identifica Ineficiência: Cria uma regra onde apenas lâmpadas de Sódio e Mercúrio contam para o "ganho" potencial com LED.

Consolida a Informação: Junta os dois ficheiros num só (df_final). Agora, cada linha é um concelho único com toda a informação necessária.

#### CÁLCULO DAS MÉTRICAS DE VIABILIDADE

No teu artigo científico, esta parte será descrita na secção de "Metodologia" ou "Processamento de Dados"

In [13]:
# 1. Ganho LED (Delta P_LED)
# Representa a potência libertada ao trocar lâmpadas de Sódio/Mercúrio por LED (65% de poupança)
df_final['Delta_P_LED'] = df_final['P_IP_Inef_kW'] * 0.65

# 2. Folga da Rede (P_Folga)
# É a capacidade que os postos já têm livre, considerando uma margem de segurança de 92%
# Fórmula: (Capacidade Total * 0.92) * (1 - Utilização Média)
df_final['P_Folga'] = (df_final['Cap_PTD_kVA'] * 0.92) * (1 - df_final['Util_Media'])

# 3. Necessidade para Veículos Elétricos (P_VE)
# Projeção: instalar carregadores de 22kW em 60% dos postos (coeficiente de simultaneidade)
df_final['P_VE'] = df_final['N_PTDs'] * 22 * 0.60

# 4. Saldo Final de Viabilidade (D)
# O indicador principal: se D > 0, o concelho tem energia suficiente para os VE
df_final['D'] = df_final['P_Folga'] + df_final['Delta_P_LED'] - df_final['P_VE']

# 5. Classificação de Viabilidade
# Criamos uma coluna booleana para facilitar filtros e cores em gráficos
df_final['Viavel'] = df_final['D'] > 0

# Visualizar os resultados para os primeiros concelhos
display(df_final[['Concelho', 'Delta_P_LED', 'P_Folga', 'P_VE', 'D', 'Viavel']].head())

,Concelho,Delta_P_LED,P_Folga,P_VE,D,Viavel
0,Águeda,158.80800,50808.195148,5121.6,45845.403148,True
1,Albergaria-a-Velha,18.21300,26596.303834,2560.8,24053.716834,True
2,Anadia,47.80425,23387.762452,2943.6,20491.966702,True
3,Arouca,75.21800,18211.314133,3115.2,15171.332133,True
4,Aveiro,93.87300,95298.999935,6718.8,88674.072935,True


##### O que significa não haver concelhos com $D < 0$?
1. Significa que, tecnicamente, todos os concelhos analisados têm "folga" suficiente na rede elétrica para suportar a instalação de carregadores de 22kW (seguindo as projeções do professor), especialmente após a poupança gerada pelos LEDs.2.
2. O conceito de "Zonas Críticas"Mesmo que todos sejam positivos, alguns estão muito mais próximos do limite do que outros.Repara no teu resultado: Albergaria-a-Velha tem um $D \approx 53.7$, enquanto Águeda tem $D \approx 845.4$.Albergaria-a-Velha é, portanto, um concelho muito mais "crítico". Se a taxa de utilização da rede subisse ligeiramente, ele passaria a ser inviável ($D < 0$).